# 02 - Attention maths
Walk through `softmax(QK^T / sqrt(d_k)) V` on a 3-token toy example, then on a real mRNA sequence.

> Student engineering project - not clinical guidance.

In [ ]:
import sys, pathlib
ROOT = pathlib.Path.cwd()
ROOT = ROOT.parent if ROOT.name == 'notebooks' else ROOT
sys.path.insert(0, str(ROOT))
import numpy as np, matplotlib.pyplot as plt
from src.attention import encode_sequence, self_attention, softmax, plot_attention, multi_head_attention

### Hand-checkable 3-token example
Tokens use 2-d vectors with Q = K = V = X (identity projections).

In [ ]:
X = np.array([[1., 0.], [0., 1.], [1., 1.]])
scores = X @ X.T / np.sqrt(2)
print('scores:\n', scores.round(3))
W = softmax(scores)
print('weights:\n', W.round(3), '\nrow sums:', W.sum(1))
print('output:\n', (W @ X).round(3))
# By hand, row 0: scores = [1, 0, 1]/sqrt(2) = [0.707, 0, 0.707]
# exp = [2.028, 1, 2.028], sum = 5.056 -> weights = [0.401, 0.198, 0.401]

In [ ]:
seq = 'ACGUACGUACGGCUA'
w, out = self_attention(encode_sequence(seq))
plot_attention(w, seq, None); plt.show()
print('uniform would be', round(1 / len(seq), 4), '| min', w.min().round(4), '| max', w.max().round(4))

### Interpretation
The weight matrices are random and scaled by 0.1, so scores are tiny and softmax is close to uniform (every weight is about 1/15 = 0.067). The visible structure comes only from the embeddings: positions holding the same base have **identical rows** and columns, because plain self-attention has no notion of position. Nothing here is learned, so the pattern shows the mechanics, not biology. Adding sinusoidal positional encoding (below) breaks the symmetry.

In [ ]:
w_pos, _ = self_attention(encode_sequence(seq, add_positions=True))
print('rows 0 and 4 (both A) identical without positions:', np.allclose(w[0], w[4]))
print('rows 0 and 4 identical with positions:', np.allclose(w_pos[0], w_pos[4]))
heads, concat = multi_head_attention(encode_sequence(seq), n_heads=4, d_k=4)
print(len(heads), 'heads; concatenated output', concat.shape)